# NeoBERT Flag Classifier (Hugging Face)

This notebook trains a binary classifier for `flag` using combined text from `name` and `description` in `filter_training.csv`.

Pipeline:
1. Setup and imports
2. Load and prepare data
3. Create train / held-out split
4. Load tokenizer and model
5. Tokenize datasets
6. Train
7. Evaluate and inspect mistakes
8. Save model artifacts

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
CSV_PATH = Path("filter_training.csv")
OUTPUT_DIR = Path("./models/flag-classifier")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CANDIDATES = [
    #"chandar-lab/NeoBERT",
    #"neobert-base",
        "answerdotai/ModernBERT-base",
    "answerdotai/ModernBERT-large",
    "bert-base-uncased",  # fallback if NeoBERT IDs are unavailable
]


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_label(value) -> int:
    return int(str(value).strip().lower() in {"true", "1", "yes", "y", "t"})


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    tp = int(((preds == 1) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())

    total = max(len(labels), 1)
    accuracy = (tp + tn) / total
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def load_first_available_model(candidates):
    last_error = None
    for model_name in candidates:
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=2,
                ignore_mismatched_sizes=True,
            )
            return model_name, tokenizer, model
        except Exception as exc:
            last_error = exc
            print(f"Could not load {model_name}: {exc}")

    raise RuntimeError(
        "None of the model candidates could be loaded. "
        f"Last error: {last_error}"
    )


set_seed(SEED)
print(f"Using torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Using torch 2.12.0
CUDA available: False


In [2]:
# Load and prepare data
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")

raw_df = pd.read_csv(CSV_PATH, delimiter="\t")
required_cols = {"name", "description", "flag"}
missing = required_cols.difference(raw_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

work_df = raw_df[["name", "description", "flag"]].copy()
work_df["name"] = work_df["name"].fillna("").astype(str)
work_df["description"] = work_df["description"].fillna("").astype(str)
work_df = work_df[(work_df["name"].str.len() > 0) | (work_df["description"].str.len() > 0)]

work_df["text"] = "title: " + work_df["name"] + " description: " + work_df["description"]
work_df["label"] = work_df["flag"].apply(to_label).astype(int)
work_df = work_df[["text", "label"]].reset_index(drop=True)

work_df = work_df.sample(frac=1)

print(f"Total rows used: {len(work_df)}")
print("Label counts:\n", work_df["label"].value_counts().sort_index())
work_df.head(3)

Total rows used: 1439
Label counts:
 label
0    1030
1     409
Name: count, dtype: int64


,text,label
168,title: Patrick Ball (actor) description: Ameri...,0
605,title: Soviet Union description: Country in Eu...,0
548,title: Adam Back description: British cryptogr...,0


In [3]:
# Create stratified train / held-out split
dataset = Dataset.from_pandas(work_df, preserve_index=False)
dataset = dataset.class_encode_column("label")
splits = dataset.train_test_split(
    test_size=0.2,
    seed=SEED,
    stratify_by_column="label",
)

train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train size: {len(train_ds)}")
print(f"Held-out size: {len(eval_ds)}")

Stringifying the column:   0%|          | 0/1439 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/1439 [00:00<?, ? examples/s]

Train size: 1151
Held-out size: 288


In [4]:
# Load tokenizer and model
selected_model_name, tokenizer, model = load_first_available_model(MODEL_CANDIDATES)
print(f"Using model: {selected_model_name}")


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using model: answerdotai/ModernBERT-base


In [5]:
# Tokenize datasets and keep Trainer-required columns
train_tok = train_ds.map(tokenize_batch, batched=True)
eval_tok = eval_ds.map(tokenize_batch, batched=True)

keep_cols = {"input_ids", "attention_mask", "label"}
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep_cols])
eval_tok = eval_tok.remove_columns([c for c in eval_tok.column_names if c not in keep_cols])

print(train_tok)
print(eval_tok)

Map:   0%|          | 0/1151 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 1151
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 288
})


In [6]:
#### Configure training and train
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.219924,0.121453,0.958333,0.972973,0.878049,0.923077
2,0.066083,0.078533,0.972222,0.951220,0.951220,0.951220
3,0.008965,0.090406,0.982639,0.987342,0.951220,0.968944
4,0.002467,0.102434,0.979167,0.975000,0.951220,0.962963
5,0.008485,0.101441,0.979167,0.975000,0.951220,0.962963


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/derenrich/.cache/uv/archive-v0/IoiiY86_VILsnkO4HQBl1/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=360, training_loss=0.09363137541157711, metrics={'train_runtime': 161.8773, 'train_samples_per_second': 35.552, 'train_steps_per_second': 2.224, 'total_flos': 90662218862964.0, 'train_loss': 0.09363137541157711, 'epoch': 5.0})

In [9]:
# Evaluate on held-out set, inspect mistakes, and save artifacts
eval_dataset =  train_tok#eval_tok #train_tok # eval_tok
eval_texts = train_ds["text"]#eval_ds["text"] # train_ds["text"] # eval_ds["text"]

metrics = trainer.evaluate(eval_dataset=eval_dataset)
print("\nHeld-out metrics:")
for k, v in metrics.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

pred_out = trainer.predict(eval_dataset)
preds = np.argmax(pred_out.predictions, axis=-1)
labels = np.array(pred_out.label_ids)
wrong_idx = np.where(preds != labels)[0]

print(f"\nMisclassified held-out examples: {len(wrong_idx)}")
if len(wrong_idx) > 0:
    for i in wrong_idx[:10]:
        print("-" * 80)
        print(f"pred={preds[i]} true={labels[i]}")
        print(eval_texts[i][:400])

save_path = OUTPUT_DIR / "best"
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))
print(f"\nSaved model and tokenizer to: {save_path.resolve()}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.008485,0.008509,5,0.998262,1.000000,0.993884,0.996933



Held-out metrics:
eval_loss: 0.0085
eval_accuracy: 0.9983
eval_precision: 1.0000
eval_recall: 0.9939
eval_f1: 0.9969



Misclassified held-out examples: 2
--------------------------------------------------------------------------------
pred=0 true=1
title: Kanye West description: American rapper and producer (born 1977)
--------------------------------------------------------------------------------
pred=0 true=1
title: D4vd description: American singer-songwriter (born 2005)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved model and tokenizer to: /Users/derenrich/projects/moonflower/models/flag-classifier/best


In [11]:
# Publish the saved model to Hugging Face Hub
import os
from huggingface_hub import HfApi, login
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HF_REPO_ID = "derenrich/wiki-modernbert-sensitive-classifier"

save_path = OUTPUT_DIR / "best"
if not save_path.exists():
    raise FileNotFoundError(
        f"Saved model folder not found at {save_path.resolve()}. Run the training/eval cell first."
    )

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN not found in environment; starting interactive Hugging Face login...")
    login()

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)

hub_model = AutoModelForSequenceClassification.from_pretrained(save_path)
hub_tokenizer = AutoTokenizer.from_pretrained(save_path)

hub_model.push_to_hub(HF_REPO_ID)
hub_tokenizer.push_to_hub(HF_REPO_ID)

print(f"Uploaded model and tokenizer to https://huggingface.co/{HF_REPO_ID}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploaded model and tokenizer to https://huggingface.co/derenrich/wiki-neobert-sensitive-classifier
